In [1]:
import os
import tensorflow as tf

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"claudesistemas","key":"a777dbe64b88859696c89ae59a328930"}')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ GPU: {tf.config.list_physical_devices('GPU')}")

print("📥 Descargando dataset cervical...")
os.system('kaggle datasets download -d paultimothymooney/cervical-cancer-risk-classification -p /content/data_cervical --unzip')
print("✅ Descarga completa")

for root, dirs, files in os.walk('/content/data_cervical'):
    level = root.replace('/content/data_cervical', '').count(os.sep)
    if level < 5:
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/ ({len(files)} archivos)")

✅ TensorFlow: 2.20.0
✅ GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
📥 Descargando dataset cervical...
✅ Descarga completa


In [6]:
for root, dirs, files in os.walk('/content/data_cervical'):
    level = root.replace('/content/data_cervical', '').count(os.sep)
    if level < 6:
        indent = ' ' * 2 * level
        n_files = len(files)
        if n_files > 0 or level < 3:
            print(f"{indent}{os.path.basename(root)}/ ({n_files} archivos)")

In [7]:
print("📥 Descargando SIPaKMeD...")
os.system('kaggle datasets download -d prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed -p /content/data_cervical2 --unzip')
print("✅ Descarga completa")

for root, dirs, files in os.walk('/content/data_cervical2'):
    level = root.replace('/content/data_cervical2', '').count(os.sep)
    if level < 5:
        indent = ' ' * 2 * level
        n_files = len(files)
        if n_files > 0 or level < 3:
            print(f"{indent}{os.path.basename(root)}/ ({n_files} archivos)")

📥 Descargando SIPaKMeD...
✅ Descarga completa
data_cervical2/ (0 archivos)
  im_Dyskeratotic/ (0 archivos)
    im_Dyskeratotic/ (1849 archivos)
      CROPPED/ (2439 archivos)
  im_Metaplastic/ (0 archivos)
    im_Metaplastic/ (1857 archivos)
      CROPPED/ (2379 archivos)
  im_Koilocytotic/ (0 archivos)
    im_Koilocytotic/ (1888 archivos)
      CROPPED/ (2475 archivos)
  im_Superficial-Intermediate/ (0 archivos)
    im_Superficial-Intermediate/ (1788 archivos)
      CROPPED/ (2493 archivos)
  im_Parabasal/ (0 archivos)
    im_Parabasal/ (1682 archivos)
      CROPPED/ (2361 archivos)


In [8]:
import tensorflow as tf
import numpy as np
import os
import shutil
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ GPU: {tf.config.list_physical_devices('GPU')}")

# ============================================================
# 1. ORGANIZAR IMÁGENES
# ============================================================
BASE = '/content/data_cervical2'
OUTPUT = '/content/dataset_cervical_procesado'

if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)

CLASES_ANORMAL = ['im_Dyskeratotic', 'im_Koilocytotic']
CLASES_NORMAL  = ['im_Metaplastic', 'im_Parabasal', 'im_Superficial-Intermediate']

def recolectar_cropped(clase_path):
    imagenes = []
    cropped_path = os.path.join(clase_path, 'CROPPED')
    if os.path.exists(cropped_path):
        for f in os.listdir(cropped_path):
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                imagenes.append(os.path.join(cropped_path, f))
    return imagenes

normales  = []
anormales = []

for clase in CLASES_NORMAL:
    path = os.path.join(BASE, clase, clase)
    imgs = recolectar_cropped(path)
    normales.extend(imgs)
    print(f"Normal — {clase}: {len(imgs)}")

for clase in CLASES_ANORMAL:
    path = os.path.join(BASE, clase, clase)
    imgs = recolectar_cropped(path)
    anormales.extend(imgs)
    print(f"Anormal — {clase}: {len(imgs)}")

print(f"\n✅ Total Normales: {len(normales)} | Anormales: {len(anormales)}")

# Split 70/15/15
n_train, n_temp = train_test_split(normales,  test_size=0.30, random_state=42)
n_val,   n_test = train_test_split(n_temp,    test_size=0.50, random_state=42)
a_train, a_temp = train_test_split(anormales, test_size=0.30, random_state=42)
a_val,   a_test = train_test_split(a_temp,    test_size=0.50, random_state=42)

splits = {
    'train':      {'normal': n_train, 'anormal': a_train},
    'validation': {'normal': n_val,   'anormal': a_val},
    'test':       {'normal': n_test,  'anormal': a_test},
}

for split, clases in splits.items():
    for clase, archivos in clases.items():
        dest = f'{OUTPUT}/{split}/{clase}'
        os.makedirs(dest, exist_ok=True)
        for src in archivos:
            shutil.copy2(src, dest)

print("\n✅ Dataset organizado:")
for split in ['train', 'validation', 'test']:
    for clase in ['normal', 'anormal']:
        n = len(os.listdir(f'{OUTPUT}/{split}/{clase}'))
        print(f"   {split}/{clase}: {n}")

# ============================================================
# 2. GENERADORES
# ============================================================
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    f'{OUTPUT}/train', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED
)
val_gen = val_test_datagen.flow_from_directory(
    f'{OUTPUT}/validation', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED
)
test_gen = val_test_datagen.flow_from_directory(
    f'{OUTPUT}/test', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)

print(f"\n✅ Clases: {train_gen.class_indices}")
print(f"✅ Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}")

# ============================================================
# 3. CLASS WEIGHTS
# ============================================================
labels = train_gen.classes
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weight_dict = dict(enumerate(class_weights))
print(f"\n✅ Class weights: {class_weight_dict}")

# ============================================================
# 4. MODELO
# ============================================================
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
print(f"\n✅ Modelo creado: {model.count_params():,} parámetros")

# ============================================================
# 5. FASE 1
# ============================================================
print("\n🚀 FASE 1 — Entrenando cabeza...")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_f1 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

model.fit(
    train_gen, validation_data=val_gen,
    epochs=20, callbacks=callbacks_f1,
    class_weight=class_weight_dict, verbose=1
)

# ============================================================
# 6. FASE 2
# ============================================================
print("\n🚀 FASE 2 — Fine tuning últimas 30 capas...")

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_f2 = [
    ModelCheckpoint('mejor_modelo_cervical_v2.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7, verbose=1),
]

model.fit(
    train_gen, validation_data=val_gen,
    epochs=50, callbacks=callbacks_f2,
    class_weight=class_weight_dict, verbose=1
)

# ============================================================
# 7. EVALUACIÓN
# ============================================================
print("\n📊 Evaluando en test set...")
test_gen.reset()
results = model.evaluate(test_gen, verbose=1)
accuracy = results[1] * 100

print(f"""
╔════════════════════════════════════════╗
║  PRECISIÓN FINAL: {accuracy:.2f}%{' ' * (18 - len(f'{accuracy:.2f}'))}║
╚════════════════════════════════════════╝
""")

# ============================================================
# 8. GUARDAR Y DESCARGAR
# ============================================================
model.save('modelo_cervical_v2.keras')

from google.colab import files
files.download('mejor_modelo_cervical_v2.keras')
print("📥 Descargando mejor modelo...")

✅ TensorFlow: 2.20.0
✅ GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Normal — im_Metaplastic: 793
Normal — im_Parabasal: 787
Normal — im_Superficial-Intermediate: 831
Anormal — im_Dyskeratotic: 813
Anormal — im_Koilocytotic: 825

✅ Total Normales: 2411 | Anormales: 1638

✅ Dataset organizado:
   train/normal: 1212
   train/anormal: 937
   validation/normal: 335
   validation/anormal: 239
   test/normal: 344
   test/anormal: 232
Found 2149 images belonging to 2 classes.
Found 574 images belonging to 2 classes.
Found 576 images belonging to 2 classes.

✅ Clases: {'anormal': 0, 'normal': 1}
✅ Train: 2149 | Val: 574 | Test: 576

✅ Class weights: {0: np.float64(1.146744930629669), 1: np.float64(0.8865511551155115)}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

✅ Modelo creado: 4,415,652 parámetros

🚀 FASE 1 — Entrenando cabeza...
Epoch 1/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 94s 934ms/step - accuracy: 0.8125 - loss: 0.4437 - val_accuracy: 0.8659 - val_loss: 0.3346 - 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Descargando mejor modelo...
